# Phase 1 — OCR Foundation and Preprocessing Research

**Project:** VIGILOX Document Intelligence
**Phase Status:** Complete ✅
**Objective:** Establish a reliable OCR foundation for extracting text, OCR confidence, and document layout information from security and identity documents, and experimentally determine whether image preprocessing improves OCR quality.

---

## 1. Phase 1 Objective

The purpose of Phase 1 was to build and evaluate the OCR layer of the VIGILOX Document Intelligence system.

The OCR layer is responsible for converting a document image into machine-readable information that can later be used by structured extraction and validation layers.

The required OCR output for each detected text line was:

* Extracted text
* OCR confidence score
* Bounding box coordinates
* Line ordering

The phase also investigated whether common image preprocessing techniques improve OCR accuracy.

The target Phase 1 pipeline was:

```text
Document Image
      ↓
Image Preprocessing Experiments
      ↓
PaddleOCR
      ↓
Text + Confidence + Bounding Boxes
      ↓
OCR Evaluation
```

Three representative document types were used during the initial research:

1. SIA security badge
2. National identity card
3. Private security guard licence

---

# 2. Development Environment

Phase 1 was developed and tested on Windows using PowerShell and a Python virtual environment.

### Environment

| Component                  | Version / Configuration             |
| -------------------------- | ----------------------------------- |
| Operating System           | Windows                             |
| Shell                      | PowerShell                          |
| Python                     | 3.13.9                              |
| PaddlePaddle               | 3.3.1                               |
| PaddleOCR                  | 3.7.0                               |
| Execution Device           | CPU                                 |
| OCR Language Configuration | English                             |
| OpenCV                     | Used for preprocessing              |
| NumPy                      | Used for numerical/image operations |

The project uses an isolated Python virtual environment named:

```text
.venv
```

This keeps the project dependencies separate from the global Python installation.

---

# 3. Initial Project Setup

The project directory was created and a Python virtual environment was configured.

Example project location:

```text
C:\Users\DELL\Desktop\Intern\VIGILOX-Document-Intelligence
```

The virtual environment was created using:

```powershell
python -m venv .venv
```

It was then activated using:

```powershell
.\.venv\Scripts\Activate.ps1
```

After activation, the PowerShell prompt showed:

```text
(.venv)
```

indicating that the project environment was active.

The Python version was checked with:

```powershell
python --version
```

The working environment used:

```text
Python 3.13.9
```

---

# 4. Core Phase 1 Libraries

## 4.1 PaddlePaddle

PaddlePaddle provides the deep-learning runtime used by PaddleOCR.

It was installed using:

```powershell
python -m pip install paddlepaddle==3.3.1
```

The installation was verified with:

```powershell
python -c "import paddle; print(paddle.__version__)"
```

A deeper runtime verification was performed using:

```powershell
python -c "import paddle; paddle.utils.run_check()"
```

The final verification result confirmed:

```text
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully!
```

This confirmed that PaddlePaddle was operational in CPU mode.

---

## 4.2 PaddleOCR

PaddleOCR was selected as the OCR engine because it provides:

* Text detection
* Text recognition
* Confidence scores
* Bounding boxes
* Text-line orientation handling
* Strong performance on document images

PaddleOCR was installed using:

```powershell
python -m pip install paddleocr==3.7.0
```

The installation was verified using:

```powershell
python -c "import paddleocr; print(paddleocr.__version__)"
```

The Phase 1 environment used:

```text
PaddleOCR 3.7.0
```

---

## 4.3 OpenCV

OpenCV was used for image preprocessing experiments, including:

* Grayscale conversion
* Otsu thresholding
* Edge detection
* Hough-line detection
* Rotation
* Deskewing
* Saving preprocessing variants

---

## 4.4 NumPy

NumPy was used for:

* Image-array manipulation
* Angle calculations
* Median skew estimation
* Numerical operations required by OpenCV processing

---

# 5. OCR Models

During OCR initialization, PaddleOCR loaded the following models:

```text
PP-LCNet_x1_0_textline_ori
PP-OCRv6_medium_det
PP-OCRv6_medium_rec
```

Their responsibilities are approximately:

| Model                          | Purpose                      |
| ------------------------------ | ---------------------------- |
| PP-LCNet text-line orientation | Detect text-line orientation |
| PP-OCRv6 medium detection      | Detect text regions          |
| PP-OCRv6 medium recognition    | Recognize detected text      |

The downloaded models were cached locally under the PaddleX model directory, so they did not need to be downloaded on every execution.

Example cache location:

```text
C:\Users\DELL\.paddlex\official_models\
```

---

# 6. PaddleOCR Compatibility Issue

During initial setup, PaddleOCR produced a Paddle/PIR oneDNN-related runtime error similar to:

```text
NotImplementedError:
ConvertPirAttribute2RuntimeAttribute not support
[pir::ArrayAttribute<pir::DoubleAttribute>]
```

The OCR runtime was stabilized by disabling MKLDNN/oneDNN acceleration for the OCR configuration.

The resulting configuration successfully operated on CPU.

This was an important environment-specific finding because the OCR models themselves were functional; the problem was related to the optimized execution backend rather than document processing logic.

---

# 7. Phase 1 Test Documents

Three sample documents were selected to represent different OCR conditions.

| Document      | Approx. Size | Characteristics                                             |
| ------------- | -----------: | ----------------------------------------------------------- |
| SIA Badge     |    285 × 177 | Security licence/badge with horizontal and vertical text    |
| ID Card       |    200 × 140 | Very low-resolution multilingual identity document          |
| Guard Licence |    563 × 355 | Larger English security licence with many structured fields |

The files were stored under:

```text
samples/
├── sia_badge.jpg
├── id_card.jpg
└── guard_license.jpg
```

Using three different document types helped expose different OCR failure modes instead of optimizing only for a single document.

---

# 8. Baseline OCR Experiment

The baseline experiment used the original document images without intentional preprocessing.

The OCR test was run using:

```powershell
python test_ocr.py
```

Each OCR line was displayed with:

* Line number
* Recognized text
* OCR confidence
* Simple confidence-based status

The initial experimental confidence threshold was:

```text
Confidence ≥ 90% → OK
Confidence < 90% → REVIEW
```

This threshold was used only for observing OCR behavior. It was **not treated as proof that a value was correct**.

---

# 9. Baseline OCR Results — SIA Badge

The SIA badge produced:

```text
1099 4265 1706 9065               98.62%
LICENCE                           99.99%
EITY                              84.27%
Security Industry Authority       99.98%
EXPIRES                           99.84%
24 MAR 2021                       99.99%
M.GREEN                           96.43%
24/03/21                          99.96%
```

## Observations

The important business fields were recognized very well:

* Licence number: correct
* Issuer: correct
* Expiry label: correct
* Expiry date: correct
* Name: correct

One decorative/vertical text region was recognized incorrectly as:

```text
EITY
```

with approximately:

```text
84.27%
```

confidence.

This demonstrated that document layout and rotated/vertical text can still cause recognition problems even when the primary business fields are recognized correctly.

### SIA baseline conclusion

The SIA badge was considered strong enough for downstream structured extraction because the critical fields were correctly detected.

---

# 10. Baseline OCR Results — ID Card

The identity card produced results including:

```text
CAN CUÓC CONG DAN                 92.14%
026205000366                      100.00%
PHAN VAN MANH                     95.29%
23/08/2006                        88.92%
```

Several other multilingual lines had substantially lower confidence.

## Important document characteristic

The source image measured only:

```text
200 × 140 pixels
```

This is extremely small for an identity document.

As a result, several text regions contained very few pixels per character.

### Critical fields

Despite the poor source resolution:

* ID number was correctly recognized
* Name was correctly recognized
* Date of birth was correctly recognized

The DOB:

```text
23/08/2006
```

was correct even though its confidence was only approximately:

```text
88.92%
```

This became an important lesson:

> A confidence below 90% does not automatically mean the OCR value is incorrect.

---

# 11. Baseline OCR Results — Guard Licence

The guard licence produced especially strong OCR results for its important fields.

Examples included:

```text
Texas                              100.00%
PRIVATE SECURITY LICENSE           99.26%
PRINTDATE 01/01/2025               99.90%
LICENSE                            99.97%
12345678                           100.00%
CATEGORY NSA/STG                   99.97%
EXPIRES                            99.84%
01/01/2026                         100.00%
DOB                                99.99%
01/01/1990                         100.00%
SAMPLE,JANE                        99.99%
ISSUED BY TX DPS                   98.79%
```

Some decorative or less important text regions were misrecognized or received lower confidence.

## Observation

The larger input resolution of:

```text
563 × 355
```

provided significantly stronger overall OCR quality than the very small ID card.

The important structured fields were recognized with near-perfect confidence.

---

# 12. Main Baseline OCR Finding

The three-document test showed that PaddleOCR was sufficiently capable for the project foundation.

However, it also revealed that:

```text
OCR confidence ≠ ground-truth correctness
```

A confidence score describes the OCR model's confidence in its recognition output.

It does **not** independently verify that the recognized value is factually correct.

This distinction became one of the main design principles for the project.

---

# 13. Image Preprocessing Research

After establishing the OCR baseline, several preprocessing techniques were evaluated.

The purpose was to answer:

> Should documents automatically be preprocessed before OCR?

Four variants were evaluated.

---

## 13.1 Original

The image was passed to OCR without intentional visual transformation.

This acted as the baseline against which other methods were compared.

---

## 13.2 Grayscale

The image was converted from colour into grayscale.

### Hypothesis

Removing colour information could:

* simplify the image
* improve text/background separation
* reduce unnecessary visual complexity

### Risk

Colour differences may also contain useful information for the OCR detector.

Therefore grayscale preprocessing cannot automatically be assumed to improve OCR.

---

# 14. Otsu Thresholding

The grayscale image was transformed into a binary black-and-white image using Otsu thresholding.

### Hypothesis

Strong text/background separation could make characters easier to recognize.

### Risk

Thresholding can destroy:

* fine character strokes
* anti-aliasing
* weak text
* low-resolution characters
* subtle boundaries

This risk proved important during the experiments.

---

# 15. Deskewing

Documents can be photographed or scanned with slight rotation.

Deskewing attempts to:

1. detect major document/text lines
2. estimate their angle
3. calculate the median skew
4. rotate the image in the opposite direction

The Phase 1 experiment used edge detection and Hough-line analysis to estimate small rotations.

Only relatively small line angles were considered so that vertical objects or decorative elements would not incorrectly determine document rotation.

---

# 16. OpenCV Hough-Line Compatibility Finding

During deskew implementation, different OpenCV behavior was observed in the structure returned by Hough-line detection.

Some environments can return coordinates effectively shaped like:

```text
[[x1, y1, x2, y2]]
```

while the tested environment could expose them in a flatter form such as:

```text
[x1, y1, x2, y2]
```

The preprocessing logic was therefore adjusted to normalize the detected coordinate structure before extracting the four values.

This made the deskew experiment more robust across OpenCV output shapes.

---

# 17. Running the Preprocessing Experiment

The preprocessing experiment was executed using:

```powershell
python test_preprocessing.py
```

It created four variants for every test document:

```text
original.jpg
grayscale.jpg
threshold.jpg
deskew.jpg
```

The generated files were stored under:

```text
output/preprocessing/
```

with a separate directory for each document.

---

# 18. Detected Skew Results

For the current three sample documents:

| Document      | Detected Skew |
| ------------- | ------------: |
| SIA Badge     |         0.00° |
| ID Card       |         0.00° |
| Guard Licence |         0.00° |

Therefore, the deskew version was effectively equivalent to the original version for these particular samples.

This does **not** mean deskewing is unnecessary in general.

It means:

> The current sample documents did not contain meaningful rotation requiring correction.

---

# 19. Preprocessing Benchmark

An additional evaluation benchmark was created to quantitatively compare the four preprocessing variants.

It was executed using:

```powershell
python preprocessing_benchmark.py
```

The benchmark measured two different metrics.

---

## 19.1 Critical Field Accuracy

A small set of known critical fields was selected for each document.

### SIA Badge

* Licence number
* Expiry date
* Name

### ID Card

* ID number
* Name
* Date of birth

### Guard Licence

* Licence number
* Expiry date
* Name

Critical-field accuracy measured the proportion of expected critical values successfully recovered by OCR.

Conceptually:

```text
Critical Field Accuracy
=
Correct Critical Fields
÷
Total Critical Fields
```

This metric was intentionally separated from OCR confidence.

---

## 19.2 Average OCR Confidence

The average recognition confidence across detected OCR lines was also calculated.

This was useful for comparison but was not treated as the primary accuracy metric.

---

# 20. Final Preprocessing Benchmark Results

| Document      | Variant   | Critical Field Accuracy | Avg. OCR Confidence |
| ------------- | --------- | ----------------------: | ------------------: |
| SIA Badge     | Original  |             **100.00%** |              98.02% |
| SIA Badge     | Grayscale |                  66.67% |              96.11% |
| SIA Badge     | Threshold |               **0.00%** |              96.50% |
| SIA Badge     | Deskew    |             **100.00%** |              98.02% |
| ID Card       | Original  |             **100.00%** |              81.94% |
| ID Card       | Grayscale |                  66.67% |          **85.14%** |
| ID Card       | Threshold |               **0.00%** |              68.92% |
| ID Card       | Deskew    |             **100.00%** |              81.94% |
| Guard Licence | Original  |             **100.00%** |              96.50% |
| Guard Licence | Grayscale |             **100.00%** |          **97.24%** |
| Guard Licence | Threshold |             **100.00%** |              93.14% |
| Guard Licence | Deskew    |             **100.00%** |              96.50% |

The CSV summary was saved under:

```text
output/preprocessing_benchmark/benchmark_summary.csv
```

---

# 21. SIA Preprocessing Findings

## Original

```text
Critical field accuracy: 100%
Average confidence: 98.02%
```

All critical fields were preserved.

---

## Grayscale

```text
Critical field accuracy: 66.67%
Average confidence: 96.11%
```

The name:

```text
M.GREEN
```

was no longer detected successfully.

Therefore grayscale preprocessing reduced critical-field performance.

---

## Threshold

```text
Critical field accuracy: 0%
Average confidence: 96.50%
```

Only a small amount of usable text remained.

This produced one of the clearest Phase 1 findings:

> A high OCR confidence value can be meaningless if the preprocessing step has already removed important information.

The threshold result had approximately 96.5% average confidence but failed all selected critical fields.

---

## Deskew

```text
Critical field accuracy: 100%
Average confidence: 98.02%
```

Because the detected skew was 0°, deskewing produced essentially the same result as the original.

---

# 22. ID Card Preprocessing Findings

The ID card produced the most important Phase 1 experiment.

## Original

```text
Critical field accuracy: 100%
Average confidence: 81.94%
```

The DOB was correctly recognized as:

```text
23/08/2006
```

with recognition confidence around:

```text
91.80%
```

in the benchmark run.

---

## Grayscale

Grayscale increased average OCR confidence:

```text
Original average confidence: 81.94%
Grayscale average confidence: 85.14%
```

At first glance, this could appear to be an improvement.

However, the DOB changed from:

```text
23/08/2006
```

to:

```text
23/05/2005
```

The incorrect grayscale DOB had approximately:

```text
94.19%
```

confidence.

Therefore:

```text
Correct:
23/08/2006
≈ 91.80%

Incorrect:
23/05/2005
≈ 94.19%
```

The incorrect value had **higher OCR confidence than the correct value**.

This became the strongest evidence that OCR confidence cannot be used as a standalone correctness metric.

---

## Threshold

Thresholding reduced critical-field accuracy to:

```text
0%
```

It also altered the ID number:

```text
Expected:
026205000366

OCR:
026205000368
```

This is particularly dangerous for document intelligence systems because a single-character error in an identifier can produce a completely different identity.

---

## Deskew

Since no meaningful skew was detected, deskewing retained:

```text
100%
```

critical-field accuracy and effectively matched the original result.

---

# 23. Guard Licence Preprocessing Findings

The guard licence was much more robust to preprocessing.

All four variants retained:

```text
100%
```

critical-field accuracy.

However, other non-critical fields degraded under thresholding.

For example, OCR text such as:

```text
DOB
```

could degrade into something similar to:

```text
DOE
```

and other secondary information also became noisier.

Therefore the fact that three selected critical fields remained correct does not mean thresholding improved the overall document.

Grayscale produced the highest average confidence:

```text
97.24%
```

while preserving the three benchmark critical fields.

This demonstrated another important finding:

> A preprocessing technique may work well for one document type and poorly for another.

---

# 24. Cross-Document Comparison

The experiments demonstrated that there is no universally superior preprocessing transformation.

### Original

Performed consistently across all three documents.

```text
SIA Badge       100%
ID Card         100%
Guard Licence   100%
```

---

### Grayscale

Performance depended strongly on the document.

```text
SIA Badge       66.67%
ID Card         66.67%
Guard Licence   100%
```

---

### Threshold

Highly destructive for two documents.

```text
SIA Badge       0%
ID Card         0%
Guard Licence   100%
```

---

### Deskew

Matched original performance because all three current documents had approximately 0° detected rotation.

```text
SIA Badge       100%
ID Card         100%
Guard Licence   100%
```

---

# 25. Key Phase 1 Research Finding

The most important finding from Phase 1 is:

> **OCR confidence must not be treated as ground-truth correctness.**

Several experiments demonstrated this.

### Example 1 — Incorrect but high confidence

ID-card grayscale DOB:

```text
23/05/2005
```

was incorrect but received approximately:

```text
94.19%
```

confidence.

---

### Example 2 — Correct but lower confidence

The correct original DOB:

```text
23/08/2006
```

received lower confidence.

---

### Example 3 — Information destroyed but confidence remains high

SIA thresholding produced:

```text
0% critical-field accuracy
```

while still reporting approximately:

```text
96.50%
```

average confidence on the small amount of text that survived.

These results demonstrate that confidence describes the model's certainty about the text it recognized, not whether preprocessing caused missing, altered, or incorrectly recognized information.

---

# 26. Final Phase 1 Preprocessing Policy

Based on the experiments, the following preprocessing policy was selected.

## Original Image — Default ✅

The original image should be the default OCR input.

Reason:

* Most consistent results
* Preserved all selected critical fields
* Avoids unnecessary image transformations
* Prevents preprocessing-induced OCR errors

---

## Deskew — Conditional ✅

Deskewing should only be applied when a meaningful rotation is actually detected.

It should not automatically rotate every document.

Current samples:

```text
Detected skew = 0.00°
```

so deskewing was unnecessary for them.

---

## Grayscale — Optional / Fallback ⚠️

Grayscale should not be applied automatically.

It improved some results on the guard licence but damaged critical-field performance on the SIA badge and ID card.

Therefore it can later be considered as:

* an alternative OCR attempt
* a fallback preprocessing variant
* an evaluation candidate

rather than the default input.

---

## Global Otsu Threshold — Rejected as Default ❌

Global thresholding should not be applied automatically.

It produced:

```text
SIA Badge → 0% critical-field accuracy
ID Card   → 0% critical-field accuracy
```

while also creating misleading confidence values.

It may still have value for particular controlled document types, but the Phase 1 evidence does not justify using it as the general preprocessing strategy.

---

# 27. Final Phase 1 OCR Strategy

The resulting OCR strategy is:

```text
Incoming Document
        ↓
Use Original Image
        ↓
Check Orientation / Skew
        ↓
Meaningful skew?
   ┌───────────┴───────────┐
   │                       │
  Yes                      No
   │                       │
Conditional Deskew      Keep Original
   │                       │
   └───────────┬───────────┘
               ↓
           PaddleOCR
               ↓
Text + Confidence + Bounding Boxes
```

Grayscale and threshold variants are not part of the default path.

---

# 28. Phase 1 Design Principles

The following principles were established for the rest of the project.

### 1. Preserve the source whenever possible

Every transformation can potentially remove useful document information.

Therefore preprocessing should be justified rather than applied automatically.

### 2. Accuracy is more important than confidence

A 99% confidence score is not useful if the recognized identifier or date is wrong.

### 3. Critical fields require separate evaluation

Average OCR metrics can hide failures in the fields that matter most.

For document intelligence systems, fields such as:

* Licence number
* ID number
* Name
* DOB
* Expiry date

must be evaluated individually.

### 4. Preprocessing must be document-aware

Different documents respond differently to the same transformation.

There is no evidence from Phase 1 supporting a universal preprocessing pipeline.

### 5. Low-resolution documents require special caution

The 200 × 140 ID card demonstrated that very small source documents significantly increase OCR uncertainty.

Image quality should therefore be considered part of later reliability decisions.

### 6. OCR output needs downstream validation

OCR alone cannot determine whether recognized information is semantically correct.

This Phase 1 finding directly motivated the later extraction and evidence-validation layers.

---

# 29. Phase 1 Limitations

The current results are **preliminary research findings**, not a final production benchmark.

Only three sample documents were used:

```text
1 SIA badge
1 ID card
1 guard licence
```

Therefore conclusions such as:

```text
Original image performs best
```

should currently be interpreted as:

> The original image was the safest default among the preprocessing techniques tested on the current sample set.

A larger labelled dataset will eventually be necessary to evaluate:

* different cameras
* lighting conditions
* blur
* perspective distortion
* document wear
* shadows
* glare
* different languages
* different licence layouts
* different image resolutions
* partial documents
* rotated documents

No production-wide accuracy claim should be made from only these three documents.

---

# 30. Phase 1 Outputs

At the end of Phase 1, the project successfully produced:

### OCR output

For every recognized line:

```text
Text
OCR confidence
Bounding box
Line ordering
```

### Preprocessing outputs

```text
Original
Grayscale
Threshold
Deskew
```

### Benchmark outputs

```text
Critical-field accuracy
Average OCR confidence
CSV benchmark summary
```

### Research conclusions

A documented preprocessing policy was established based on experimental results rather than assumptions.

---

# 31. Phase 1 Completion Checklist

| Task                                       | Status     |
| ------------------------------------------ | ---------- |
| Python virtual environment                 | ✅ Complete |
| PaddlePaddle installation                  | ✅ Complete |
| PaddlePaddle CPU verification              | ✅ Complete |
| PaddleOCR installation                     | ✅ Complete |
| OCR model loading                          | ✅ Complete |
| oneDNN/MKLDNN compatibility issue resolved | ✅ Complete |
| OCR text extraction                        | ✅ Complete |
| OCR confidence extraction                  | ✅ Complete |
| Bounding-box extraction                    | ✅ Complete |
| SIA badge OCR test                         | ✅ Complete |
| ID-card OCR test                           | ✅ Complete |
| Guard-licence OCR test                     | ✅ Complete |
| Grayscale preprocessing                    | ✅ Complete |
| Otsu threshold experiment                  | ✅ Complete |
| Deskew experiment                          | ✅ Complete |
| Hough-line compatibility issue resolved    | ✅ Complete |
| Preprocessing outputs saved                | ✅ Complete |
| Critical-field benchmark                   | ✅ Complete |
| Confidence comparison                      | ✅ Complete |
| Benchmark CSV generated                    | ✅ Complete |
| Default preprocessing decision             | ✅ Complete |

---

# 32. Phase 1 Final Conclusion

Phase 1 successfully established the OCR foundation for VIGILOX Document Intelligence.

PaddleOCR was able to recover the major business-critical fields from all three initial document types, while also providing confidence scores and bounding boxes required by later stages.

The preprocessing experiments demonstrated that additional image processing is **not automatically beneficial**.

The strongest result was observed on the low-resolution ID card, where grayscale preprocessing increased OCR confidence while changing the correct DOB:

```text
23/08/2006
```

into the incorrect value:

```text
23/05/2005
```

This confirms that OCR confidence must not be treated as a substitute for correctness.

The final Phase 1 decision is therefore:

> **Use the original document image as the default OCR input. Apply preprocessing only when there is evidence that it is required. Evaluate critical-field correctness separately from OCR confidence, and never assume that a higher confidence score means a more accurate document extraction.**

With these decisions established, **Phase 1 is complete**.
